In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
import re

In [12]:
data_root = '/astrum/home/hpchzy/code/data/'

date_base = '20260509'

output = 'output_sampOpt_filtOpt'

kernel = 'jacobi2d5p'

# hosts = ['c920bn3', 'camd9554n2', 'cgnr6760pn2']
hosts = ['c920bn3', 'camd9554n2', 'cgnr6760pn2']

# hostname = 'cgnr6760pn2'
# hostname = 'c920bn3'

# iarr = 1024

iarr = 1024

In [13]:
folder_list = []
for host in hosts:
    folder_list.extend(glob.glob(os.path.join(data_root, f'{date_base}/{host}/{output}/{kernel}{iarr}n*{host}_filt/')))

records = []
for folder_path in folder_list:
    host_match = re.search(rf'/{date_base}/([^/]+)/{output}/', folder_path)
    host = host_match.group(1) if host_match else ''
    timer_match = re.search(rf'{kernel}{iarr}.*_(.*)_{host}', folder_path)
    if timer_match is None:
        continue
    timer = timer_match.group(1)

    tf_csv_path = glob.glob(os.path.join(folder_path, 'tf.csv'))
    if not tf_csv_path:
        continue
    tm_df = pd.read_csv(tf_csv_path[0], header=None)
    tm_df.columns = ['ns']
    tm_df['timer'] = timer
    tm_df['host'] = host
    records.append(tm_df)

if records:
    tm_total_df = pd.concat(records, ignore_index=True)
    tm_total_df.sort_values(by=['host', 'timer'], inplace=True)
    tm_total_df.reset_index(drop=True, inplace=True)
else:
    tm_total_df = pd.DataFrame(columns=['ns', 'timer', 'host'])

tm_total_df

,ns,timer,host
0,13761,cgt,c920bn3
1,6880,cgt,c920bn3
2,6151,cgt,c920bn3
3,7680,cgt,c920bn3
4,6301,cgt,c920bn3
...,...,...,...
170060795,2610,tsc,cgnr6760pn2
170060796,2511,tsc,cgnr6760pn2
170060797,2599,tsc,cgnr6760pn2
170060798,2479,tsc,cgnr6760pn2


In [14]:
thresholds = tm_total_df.groupby(['host', 'timer'])['ns'].quantile(0.995)
tm_total_df = tm_total_df[
    tm_total_df['ns'] <= tm_total_df.set_index(['host', 'timer']).index.map(thresholds)
].copy()

df_sampled = (
    tm_total_df.groupby('host', group_keys=False)
    .apply(lambda g: g.sample(n=min(1000000, len(g)), random_state=42))
    .reset_index(drop=True)
)
df_sampled

/tmp/ipykernel_671022/2753046730.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=min(1000000, len(g)), random_state=42))


,ns,timer,host
0,5680,cntvct,c920bn3
1,3930,cntvcto,c920bn3
2,5340,papi,c920bn3
3,6090,cntvcto,c920bn3
4,7420,papi,c920bn3
...,...,...,...
2999995,3454,tsc,cgnr6760pn2
2999996,3166,papi,cgnr6760pn2
2999997,3663,tsc,cgnr6760pn2
2999998,2803,papi,cgnr6760pn2


In [ ]:
host_name_map = {
    'c920bn3': 'Kunpeng 920B',
    'camd9554n2': 'AMD EPYC 9554',
    'cgnr6760pn2': 'Intel Xeon 6760P',
}

fig, axes = plt.subplots(1, len(hosts), figsize=(19, 6), sharey=True)

if len(hosts) == 1:
    axes = [axes]

for ax, host in zip(axes, hosts):
    host_df = df_sampled[df_sampled['host'] == host]
    sns.violinplot(
        data=host_df,
        x='ns',
        y='timer',
        palette='viridis',
        inner='quart',
        linewidth=1.0,
        density_norm='width',
        cut=0,
        ax=ax,
    )
    ax.set_title(host_name_map.get(host, host))
    ax.set_xlabel('ns')
    ax.set_ylabel('timer' if ax is axes[0] else '')
    ax.grid(True, axis='x', alpha=0.25)

sns.despine()
plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined